In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de evaluación de Machine Learning
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("📖 Cargando los conjuntos procesados desde data/processed/...")

train_path = "../data/processed/train.parquet"
test_path = "../data/processed/test.parquet"

df_train = pd.read_parquet(train_path)
df_test = pd.read_parquet(test_path)

print(f"✅ Datos cargados en memoria:")
print(f" • Conjunto de Entrenamiento: {len(df_train):,} decisiones")
print(f" • Conjunto de Prueba (Test):  {len(df_test):,} decisiones")

# Separar las variables de entrada (X) y la etiqueta objetivo (y)
features = ['calle', 'bote', 'to_call', 'stack', 'pot_odds']
target = 'accion'

X_train = df_train[features]
y_train = df_train[target]

X_test = df_test[features]
y_test = df_test[target]

print(f"\n🎯 Variables de entrada (X): {features}")
print(f"🎯 Variable a predecir   (y): '{target}' (0=Fold, 1=Check/Call, 2=Bet/Raise)")

📖 Cargando los conjuntos procesados desde data/processed/...
✅ Datos cargados en memoria:
 • Conjunto de Entrenamiento: 263,225 decisiones
 • Conjunto de Prueba (Test):  56,362 decisiones

🎯 Variables de entrada (X): ['calle', 'bote', 'to_call', 'stack', 'pot_odds']
🎯 Variable a predecir   (y): 'accion' (0=Fold, 1=Check/Call, 2=Bet/Raise)


In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Baseline 1: Clase Mayoritaria (Adivinar siempre Fold a ciegas)
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_b1 = dummy.predict(X_test)

acc_b1 = accuracy_score(y_test, y_pred_b1)
f1_b1 = f1_score(y_test, y_pred_b1, average='macro', zero_division=0)

# 2. Baseline 2: Árbol de Reglas Simples (Profundidad 3)
arbol_simple = DecisionTreeClassifier(max_depth=3, random_state=42)
arbol_simple.fit(X_train, y_train)
y_pred_b2 = arbol_simple.predict(X_test)

acc_b2 = accuracy_score(y_test, y_pred_b2)
f1_b2 = f1_score(y_test, y_pred_b2, average='macro', zero_division=0)

# 3. Construir la Tabla Comparativa Oficial Corregida
tabla_baselines = pd.DataFrame([
    {
        "Modelo / Enfoque": "Baseline 1: Clase Mayoritaria (Fold)",
        "Accuracy": f"{acc_b1 * 100:.2f}%",
        "F1-Score Macro": f"{f1_b1:.3f}",
        "Propósito Académico": "Piso mínimo absoluto (adivinar sin mirar contexto)"
    },
    {
        "Modelo / Enfoque": "Baseline 2: Árbol de Reglas Simples (Prof. 3)",
        "Accuracy": f"{acc_b2 * 100:.2f}%",
        "F1-Score Macro": f"{f1_b2:.3f}",
        "Propósito Académico": "Sentido común básico: pasar si es gratis, retirarse si cobran"
    },
    {
        "Modelo / Enfoque": "🎯 Meta Red Neuronal MLP (Paso 4)",
        "Accuracy": f"> {acc_b2 * 100:.2f}%",
        "F1-Score Macro": f"> {f1_b2:.3f}",
        "Propósito Académico": "Superar al árbol con las mismas 5 variables públicas (sin cartas)"
    }
])

print("=" * 95)
print("  TABLA COMPARATIVA DE REFERENCIA (BASELINES) — Doc. Maestro § 9.1 (Actualizada)")
print("=" * 95)
print(tabla_baselines.to_string(index=False))
print("=" * 95)

  TABLA COMPARATIVA DE REFERENCIA (BASELINES) — Doc. Maestro § 9.1 (Actualizada)
                             Modelo / Enfoque Accuracy F1-Score Macro                                               Propósito Académico
         Baseline 1: Clase Mayoritaria (Fold)   48.10%          0.217                Piso mínimo absoluto (adivinar sin mirar contexto)
Baseline 2: Árbol de Reglas Simples (Prof. 3)   64.77%          0.528     Sentido común básico: pasar si es gratis, retirarse si cobran
             🎯 Meta Red Neuronal MLP (Paso 4) > 64.77%        > 0.528 Superar al árbol con las mismas 5 variables públicas (sin cartas)


### 🧠 Análisis Crítico de los Baselines (datos corregidos, sin fuga de información)

1. **El 64.77% del Baseline 2:** Un árbol de 3 preguntas acierta casi dos de cada tres decisiones porque en el póker las jugadas rutinarias son mayoría (si la apuesta es cero casi todos pasan; ante una subida preflop la mayoría se retira). Antes de corregir la fuga de información, este mismo árbol marcaba **72.21%**: parte de ese acierto venía de la pista que dejaba el bote registrado *después* de actuar.
2. **Qué debe demostrar la Red Neuronal:** con exactamente las mismas 5 variables públicas, la red solo aporta valor si captura relaciones no lineales que el árbol no ve (proporción apuesta/bote, compromiso con el bote, diferencias entre calles).
3. **Meta científica honesta:** superar al Baseline 2 tanto en *Accuracy* como en *F1-Score Macro*. Con 5 variables y sin cartas privadas el margen esperado es pequeño; los Modelos 3 (45 variables) y 4 (cartas visibles, Pluribus) investigan cómo romper ese techo.